# Lesson 07 - Planning Design Pattern

This notebook demonstrates the **Planning Design Pattern** for AI agents using the Microsoft Agent Framework.
You will learn how to break complex travel requests into structured subtasks, assign them to specialist agents,
and execute the resulting plan — all using structured output powered by Pydantic models.


## সেটআপ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## টাস্ক ডিকম্পোজিশন

টাস্ক ডিকম্পোজিশন হল পরিকল্পনা ডিজাইন প্যাটার্নের মূল। একটি একক এজেন্টকে একটি জটিল অনুরোধ সম্পূর্ণরূপে মোকাবেলা করার পরিবর্তে,
আমরা সমস্যাটিকে ছোট, সুস্পষ্ট **সাবটাস্ক** এ ভেঙে দিই।
প্রতিটি সাবটাস্ক একটি বিশেষজ্ঞ এজেন্ট (যেমন, ফ্লাইট, হোটেল, কার্যক্রম) কে বরাদ্দ করা হয় পরিষ্কার
অগ্রাধিকার এবং নির্ভরশীলতার ক্রম অনুযায়ী।

এই পদ্ধতি কয়েকটি সুবিধা প্রদান করে:
- **পূর্নতা**: প্রতিটি সাবটাস্কের একটি একক দায়িত্ব থাকে।
- **সমান্তরালতা**: স্বাধীন সাবটাস্কগুলি একসাথে চালানো যায়।
- **নির্ভরযোগ্যতা**: ব্যর্থতাগুলি নির্দিষ্ট সাবটাস্কের মধ্যে সীমাবদ্ধ থাকে।
- **বাজেট ট্র্যাকিং**: খরচগুলি প্রতিটি সাবটাস্কের জন্য অনুমান করা হয় এবং সংযুক্ত করা হয়।


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## কাঠামোবদ্ধ আউটপুট সহ একটি পরিকল্পনা এজেন্ট তৈরি করা

পরিকল্পনা এজেন্ট একটি **ফ্রন্ট ডেস্ক সমন্বয়কারী** হিসাবে কাজ করে। উচ্চ-স্তরের একটি ভ্রমণ অনুরোধ দেওয়া হলে এটি
একটি কাঠামোবদ্ধ `TravelPlan` উৎপন্ন করে — অনুরোধটিকে উপকার্যে বিভক্ত করে, অগ্রাধিকার নির্ধারণ করে,
এবং নির্ভরশীলতাগুলি চিহ্নিত করে যাতে কনসিয়ার্জ বা নির্বাহ স্তর কাজটি সম্পাদন করতে পারে।


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## বিশেষজ্ঞ সরঞ্জাম দিয়ে একটি পরিকল্পনা কার্যকর করা

একবার সামনের ডেস্ক এজেন্ট একটি কাঠামোবদ্ধ পরিকল্পনা তৈরি করলে, **কনসিয়ার্জ এজেন্ট** এটি কার্যকর করে।
প্রত্যেক বিশেষজ্ঞ সরঞ্জাম একটি উপকাজের শ্রেণি (উড়ান, হোটেল, কার্যকলাপ) পরিচালনা করে। কনসিয়ার্জ
পরিকল্পনার উপকাজগুলি নির্ভরতার ক্রমে পুনরাবৃত্তি করে এবং প্রতিটিকে উপযুক্ত
সরঞ্জামে প্রেরণ করে।


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## সারাংশ

এই পাঠে আপনি AI এজেন্টদের জন্য **পরিকল্পনা ডিজাইন প্যাটার্ন** শিখেছেন:

1. **টাস্ক ডিকম্পোজিশন** — একটি ফ্রন্ট ডেক্স পরিকল্পনা এজেন্ট একটি জটিল ভ্রমণ অনুরোধকে
   পিড্যান্টিক মডেল ব্যবহার করে সংগঠিত উপটাস্কে ভাগ করে, প্রতিটি উপ-টাস্ককে বিশেষজ্ঞ এজেন্টকে অগ্রাধিকার
   এবং নির্ভরশীলতা সহ বরাদ্দ করে।
2. **সংগঠিত আউটপুট** — একটি `response_format` পাস করে, এজেন্টটি ফ্রি-ফর্ম টেক্সটের পরিবর্তে
   একটি যাচাইকৃত `TravelPlan` অবজেক্ট ফেরত দেয়, যা নিম্নপ্রাপ্ত প্রক্রিয়াকরণকে নির্ভরযোগ্য করে তোলে।
3. **পরিকল্পনা বাস্তবায়ন** — একটি কনসিয়ার্জ এজেন্ট বিশেষজ্ঞ সরঞ্জাম ব্যবহার করে উপ-টাস্কগুলির মধ্য দিয়ে পুনরাবৃত্তি করে
   (`book_flight`, `reserve_hotel`, `book_activity`) পরিকল্পনা সম্পাদন এবং ফলাফল প্রতিবেদন করার জন্য।

এই প্যাটার্ন *কি করতে হবে* (পরিকল্পনা) কে *কিভাবে করতে হবে* (বাস্তবায়ন) থেকে পৃথক করে, এজেন্টদের
আরও মডুলার, পরীক্ষাযোগ্য এবং সম্প্রসারণে সহজ করে তোলে।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
